In [2]:
import os
import glob
import math
import pandas as pd
import matplotlib.pyplot as plt

def create_plots_for_folders(base_folder_path):
    """
    Loops through all subfolders in the base_folder_path.
    For each folder, it reads all CSV files and creates one combined image with plots.
    """
    # Find all items in the base folder
    # os.listdir gets everything in the folder, os.path.join creates the full path
    all_items = [os.path.join(base_folder_path, item) for item in os.listdir(base_folder_path)]
    
    # Filter out only the directories (folders)
    folders = [item for item in all_items if os.path.isdir(item)]
    
    if not folders:
        print(f"Could not find any folders inside {base_folder_path}")
        return

    # Loop through each folder we found
    for folder in folders:
        print(f"Processing folder: {folder}")
        
        # Find all .csv files in this specific folder
        csv_files = glob.glob(os.path.join(folder, "*.csv"))
        
        # If there are no CSV files in this folder, skip to the next folder
        if len(csv_files) == 0:
            print(f"  No CSV files found in {folder}. Skipping.")
            continue
            
        # Calculate the grid size for the subplots (e.g., 3x3 grid for 8 files)
        num_files = len(csv_files)
        columns = math.ceil(math.sqrt(num_files))
        rows = math.ceil(num_files / columns)
        
        # Create a large figure to hold all the small graphs
        # figsize=(width, height) - we make it dynamic based on rows and columns
        fig, axes = plt.subplots(nrows=rows, ncols=columns, figsize=(columns * 6, rows * 4))
        
        # Make sure 'axes' is always a flat list so it's easier to loop through
        if num_files == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
            
        # Loop through the CSV files and the graph spaces (axes) at the same time
        for i, csv_file in enumerate(csv_files):
            ax = axes[i] # Get the specific graph area
            
            # Read the CSV file using pandas
            try:
                df = pd.read_csv(csv_file)
            except Exception as e:
                print(f"  Error reading {csv_file}: {e}")
                continue
                
            # Assume the first column is the X-axis (timestamp)
            x_column = df.columns[0]
            
            # The remaining columns are the Y-axis lines we want to plot
            y_columns = df.columns[1:]
            
            # Plot each Y column against the X column
            for y_col in y_columns:
                ax.plot(df[x_column], df[y_col], label=y_col)
                
            # Add titles, labels, and a legend (the box explaining the colors)
            file_name_only = os.path.basename(csv_file)
            ax.set_title(file_name_only, fontsize=10)
            ax.set_xlabel(x_column)
            ax.set_ylabel("Values")
            ax.legend(loc='upper right', fontsize=8)
            ax.grid(True, linestyle='--', alpha=0.6) # Add a faint grid for readability

        # If there are empty graph spaces in our grid, hide them so it looks neat
        for j in range(num_files, len(axes)):
            fig.delaxes(axes[j])
            
        # Adjust the layout so labels don't overlap
        plt.tight_layout()
        
        # Save the beautiful combined picture inside the folder!
        # We name it after the folder so it's easy to identify
        folder_name = os.path.basename(folder)
        save_path = os.path.join(folder, f"{folder_name}_combined_plots.png")
        plt.savefig(save_path, dpi=150) # dpi=150 makes it high quality
        plt.close(fig) # Close the figure to free up computer memory
        
        print(f"  Successfully saved image to: {save_path}")

BASE_DIRECTORY = "./demo_runs"

# Run the function
if __name__ == "__main__":
    create_plots_for_folders(BASE_DIRECTORY)

Processing folder: ./demo_runs/OpenWearable_Recording_2026-03-26T101935.303786_3_demo
  Successfully saved image to: ./demo_runs/OpenWearable_Recording_2026-03-26T101935.303786_3_demo/OpenWearable_Recording_2026-03-26T101935.303786_3_demo_combined_plots.png
